# Saving Brains — KMC-400-20y
## M8: WASI K=3 Predictive Model — Three-Class Cognitive Phenotype
### Phase 1 (Birth → 3 months CA) · Phase 2 (Birth → 12 months CA)

**Target:** WASI clustering with K=3 — mathematically optimal for WASI FSIQ distribution  
**Classes:** GO-1 High IQ · GO-2 Medium IQ · GO-3 Low IQ (risk)  
**Task:** Multiclass classification (softmax) + binary risk flag (GO-3 vs rest)  

---

### Key changes from M7 (binary GO-i K=2)

| Aspect | M7 — Binary | M8 — WASI K=3 |
|--------|-------------|----------------|
| Cluster source | `clusters_GOi.csv` | `clusters_GO_WASI_v1.csv` |
| Classes | GO-1 High / GO-2 Low | GO-1 High / GO-2 Medium / GO-3 Low |
| Target column | `GO_i` (1 or 2) | `GO_i` (1, 2, or 3) |
| XGBoost objective | `binary:logistic` | `multi:softmax` (n_class=3) |
| Primary metric | AUC-ROC (binary) | Macro-AUC + per-class metrics |
| Risk flag | GO-2 = risk | GO-3 = risk (binary collapse) |
| Scale pos weight | n_GO1/n_GO2 | `sample_weight` per class |
| Threshold | Single (F1-optimal) | Per-class probability threshold |

---

### Two evaluation strategies

**Strategy A — Multiclass (native K=3)**  
Predict all three classes simultaneously. Metrics: macro-AUC, weighted F1,  
confusion matrix 3×3, per-class precision/recall.

**Strategy B — Binary collapse (GO-3 vs rest)**  
Collapse GO-1 and GO-2 into "not at risk" and predict GO-3 as the positive class.  
Allows direct comparison with M7 metrics and clinical threshold analysis.  
Both strategies are run and compared in Section 10.


## 1. Setup

In [0]:
# Databricks only: uncomment to restart Python after pip installs
# dbutils.library.restartPython()
print("Environment ready.")

In [0]:
dbutils.library.restartPython()

In [0]:
# Install required packages (uncomment if not already installed)
# pip install "numpy<2" "xgboost<3" shap imbalanced-learn statsmodels
# In Databricks: %pip install "numpy<2" "xgboost<3" shap --quiet
%pip install "numpy<2" "xgboost<3" shap --quiet

In [0]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    brier_score_loss, roc_curve, confusion_matrix,
)
from xgboost import XGBClassifier
import shap
import joblib

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_theme(style="whitegrid")

import mlflow
import mlflow.xgboost

# ── Paths — adjust to your environment ───────────────────────────────────
DATA_MASTER  = "/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv"
DATA_CLUSTER = "/Volumes/workspace/default/kmc_data/clusters_GO_WASI_k3.csv"
EXP_NAME     = "/Users/a.echeverrig@uniandes.edu.co/prediccion-ci-prematuros-madre-canguro/notebooks/SavingBrains_M7_TwoPhase"

mlflow.set_registry_uri("databricks")
mlflow.set_experiment(EXP_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment         :", EXP_NAME)
print("xgboost:", __import__('xgboost').__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


## 2. Constants, Criteria & Feature Groups

In [0]:
RANDOM_STATE = 42
N_FOLDS      = 5

# ─────────────────────────────────────────────────────────────────────────
# EXPLICIT INCLUSION CRITERIA
# (expert requirement: make criteria explicit and replicable)
# ─────────────────────────────────────────────────────────────────────────
INCLUSION_CRITERIA = {
    "temporal"      : "Measured BEFORE the outcome window (before 20 years)",
    "codebook"      : "Confirmed in SPSS codebook (Feb 2024) OR expert-validated",
    "coverage"      : "Available in >= 60% of G1 preterm sample (n_valid >= 232)",
    "non_leakage"   : "Not a proxy of the cognitive outcome (GO-i) itself",
    "collinearity"  : "Pearson r < 0.85 with all other features in the same model",
    "justification" : "Has a published clinical or bibliographic rationale",
}

EXCLUSION_CRITERIA = {
    "outcome_var"   : "Measured at 20-year follow-up (WASI_, CVLT_, TAP_, NHPT_, etc.)",
    "codebook_miss" : "Not in codebook AND not expert-confirmed",
    "low_coverage"  : "Available in < 60% of sample",
    "same_construct": "Redundant with another included feature (same construct)",
    "high_corr"     : "r >= 0.85 with another feature already in model",
}

print("Inclusion criteria:")
for k, v in INCLUSION_CRITERIA.items():
    print(f"  [{k}] {v}")
print()
print("Exclusion criteria:")
for k, v in EXCLUSION_CRITERIA.items():
    print(f"  [{k}] {v}")

# ─────────────────────────────────────────────────────────────────────────
# FENTON 2013 HC REFERENCE TABLES
# Source: Fenton TR & Kim JH (2013) BMC Pediatrics 13:59
# Formula: z = (X / M - 1) / S   [LMS method, L≈1, Cole & Green 1992]
# Valid range: GA 22–41 weeks (birth and 40-week corrected age ONLY)
# Beyond 40w CA: WHO curves with corrected age (NUT12M_* variables)
# ─────────────────────────────────────────────────────────────────────────
FENTON_HC_M = {
    24:21.8, 25:23.0, 26:24.2, 27:25.4, 28:26.6, 29:27.8, 30:28.9,
    31:30.0, 32:31.0, 33:31.9, 34:32.7, 35:33.4, 36:34.1, 37:34.7,
    38:35.1, 39:35.5, 40:35.9, 41:36.1,
}
FENTON_HC_S = {
    24:.058, 25:.056, 26:.054, 27:.052, 28:.050, 29:.048, 30:.046,
    31:.045, 32:.044, 33:.043, 34:.042, 35:.041, 36:.041, 37:.040,
    38:.040, 39:.040, 40:.040, 41:.040,
}

# ─────────────────────────────────────────────────────────────────────────
# FEATURE GROUPS — PHASE 1 (Birth → 3 months corrected age)
# ─────────────────────────────────────────────────────────────────────────

# F1A: KMC intervention
# Special imputation: TC group = 0 (no kangaroo care by study design)
FEATURES_P1_KMC = [
    "EX41_durPCconcontroles03",     # KMC duration (hours) — TC imputed as 0 ***expert
]

# F1B: Neonatal severity (confirmed in codebook)
FEATURES_P1_NEONATAL_CB = [
    "NEO_ventilacion",              # Mechanical ventilation type  **
    "NEO_UCI",                      # NICU admission  *
    "NEO_diasaminoglucosidos",      # Days of aminoglycosides  **  (ototoxic: auditory risk)
    "NEO_recibioaminiglucosidos",   # Received aminoglycosides (binary)  †
]

# F1C: Neonatal severity (in dataset, expert-confirmed — not in codebook)
FEATURES_P1_NEONATAL_DS = [
    "NEO_fotote6",                  # Days of phototherapy  ***
    "NEO_HOSP",                     # Days of hospitalisation  **
    "NEO_diasventiladores",         # Days of mechanical ventilation  *
    "NEO_totoxidias",               # Days of oxygen therapy  **
    "NEO_parentNutritiondias",      # Days of parenteral nutrition  **
]

# F1D: Early neurological screening (neonatal)
# Note: APGAR p=0.38 ns in bivariate but included per expert guidance
#       as early warning signal — XGBoost may find joint effects
FEATURES_P1_NEURO = [
    "BIRTH_apgar1_5",           # APGAR score at 1 min  ns (expert: early alert)
    "BIRTH_apgar5_5",           # APGAR score at 5 min  ns (expert: early alert)
    "NEURO_leucomalaciacat",    # Periventricular leukomalacia  *  (58% data)
]

# F1E: Fenton growth — birth and 40 weeks (derived; source vars in codebook)
# F_catchup_hc_fenton and F_z_hc_birth_fenton — built in Section 4
FEATURES_P1_FENTON = [
    "F_catchup_hc_fenton",      # HC z-score change birth→40w  ***
    "F_z_hc_birth_fenton",      # HC z-score at birth  *
]

# F1F: Growth velocity (WHO z-scores, corrected age) — 40w→3m CA
# Velocity = delta z-score; avoids multicollinearity vs levels (r<0.46)
# Source: NUT12M_wam8 (41w), NUT12M_wam9 (3m CA), NIHS reference
FEATURES_P1_VELOCITY = [
    "F_delta_waz_40w_3m",       # Δ weight-for-age z 40w→3m  ns
    "F_delta_haz_40w_3m",       # Δ height-for-age z 40w→3m  ns
    "F_delta_hcz_40w_3m",       # Δ HC-for-age z 40w→3m  ns
    "NUT12M_pcam5",             # HC/expected at birth (WHO ratio ×100)  *
]

# F1G: Breastfeeding
FEATURES_P1_FEEDING = [
    "NUT12M_LM12m",             # Breastfeeding status  ns (clinical relevance)
]

# F1H: Socioeconomic context
FEATURES_P1_SES = [
    "SCB_nivm1",                # Maternal education level  **
    "SCB_nivp1",                # Paternal education level  *  (new)
    "SCB_percap1",              # Household per-capita income  ns
]

# All Phase 1 features
ALL_FEATURES_P1 = (
    FEATURES_P1_KMC +
    FEATURES_P1_NEONATAL_CB +
    FEATURES_P1_NEONATAL_DS +
    FEATURES_P1_NEURO +
    FEATURES_P1_FENTON +
    FEATURES_P1_VELOCITY +
    FEATURES_P1_FEEDING +
    FEATURES_P1_SES
)

# ─────────────────────────────────────────────────────────────────────────
# FEATURE GROUPS — PHASE 2 (adds 3m → 12m CA variables)
# Includes all Phase 1 features + the following
# ─────────────────────────────────────────────────────────────────────────

# F2A: Psychomotor development — Griffiths (6m and 12m CA)
FEATURES_P2_GRIFFITHS = [
    "PMD_RSM6",      # Raw motor score 6m  ***
    "PMD_coaudl6",   # Auditory quotient 6m  ***
    "PMD_cogrif6",   # General quotient 6m  **
    "PMD_coloco6",   # Locomotion 6m  *
    "PMD_cogrif12",  # General quotient 12m  *
    "PMD_coloco12",  # Locomotion 12m  *
    "PMD_copers12",  # Personal-social 12m  *
]

# F2B: Growth velocity — 3m→12m CA (expert: "how I got there matters")
FEATURES_P2_VELOCITY = [
    "F_delta_waz_3m_12m",   # Δ WAZ 3m→12m  *  (GO-2 shows paradoxical catch-up)
    "F_delta_haz_3m_12m",   # Δ HAZ 3m→12m  *
    "F_delta_hcz_3m_12m",   # Δ HCZ 3m→12m  ns
]

# F2C: Home environment (12m CA)
FEATURES_P2_HOME = [
    "HOMET_tsubs_tothome12mfact",     # HOME factor score  *
    "HOMET_ttotalsubscalehome12mraw", # HOME total raw  †
]

# F2D: Neurodevelopmental screening (6–12m CA)
# INFANIB: p=ns in bivariate but expert-recommended
FEATURES_P2_NEURODEV = [
    "FOLL12M_infanib9",    # INFANIB 3m CA  ns (expert: key clinical exam)
    "FOLL12M_infanib12",   # INFANIB 12m CA  ns
]

# F2E: Condition at birth (retained in Phase 2)
FEATURES_P2_BIRTH = [
    "BIRTH_peso5",   # Birth weight (g)  ns
    "EX41_talla8",   # Length at 40w CA (mm)  †
    "EX41_peso8",    # Weight at 40w CA (g)  ns
]

# All Phase 2 features = Phase 1 + additional
ALL_FEATURES_P2 = (
    ALL_FEATURES_P1 +
    FEATURES_P2_GRIFFITHS +
    FEATURES_P2_VELOCITY +
    FEATURES_P2_HOME +
    FEATURES_P2_NEURODEV +
    FEATURES_P2_BIRTH
)

# ── XGBoost hyperparameters ───────────────────────────────────────────────
XGB_PARAMS = dict(
    objective        = "binary:logistic",
    n_estimators     = 200,
    learning_rate    = 0.05,
    max_depth        = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_lambda       = 2.0,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# ── Colour palette ────────────────────────────────────────────────────────
PAL = {
    "dark_blue"  : "#1A5276",
    "mid_blue"   : "#2E86C1",
    "light_blue" : "#AED6F1",
    "red"        : "#C0392B",
    "green"      : "#1E8449",
    "orange"     : "#D35400",
    "purple"     : "#884EA0",
}

print("Phase 1 features:", len(ALL_FEATURES_P1))
print("Phase 2 features:", len(ALL_FEATURES_P2))
print("Added in Phase 2:", len(ALL_FEATURES_P2) - len(ALL_FEATURES_P1))


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# WASI K=3 — XGBoost multiclass configurations
#
# WASI K=3 classes:
#   0 → GO-1 High IQ   (label 1 in CSV → remapped to 0)
#   1 → GO-2 Medium IQ (label 2 in CSV → remapped to 1)
#   2 → GO-3 Low IQ    (label 3 in CSV → remapped to 2)
#
# Objective: multi:softmax  → predicts class labels
#            multi:softprob → predicts probability per class (needed for AUC)
#
# Class weights: applied via sample_weight in fit(), not scale_pos_weight
# (scale_pos_weight is only for binary classification)
# ─────────────────────────────────────────────────────────────────────────

# Baseline multiclass
XGB_PARAMS_MC = dict(
    objective        = "multi:softprob",
    num_class        = 3,
    n_estimators     = 200,
    learning_rate    = 0.05,
    max_depth        = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_lambda       = 2.0,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# Anti-overfitting multiclass
XGB_PARAMS_MC_REG = dict(
    objective        = "multi:softprob",
    num_class        = 3,
    n_estimators     = 150,
    learning_rate    = 0.03,
    max_depth        = 3,
    subsample        = 0.7,
    colsample_bytree = 0.7,
    min_child_weight = 8,
    reg_lambda       = 4.0,
    reg_alpha        = 0.5,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# Binary collapse: GO-3 vs rest (GO-1 + GO-2 merged)
# Allows direct comparison with M7 metrics
XGB_PARAMS_BINARY_COLLAPSE = dict(
    objective        = "binary:logistic",
    n_estimators     = 150,
    learning_rate    = 0.03,
    max_depth        = 3,
    subsample        = 0.7,
    colsample_bytree = 0.7,
    min_child_weight = 8,
    reg_lambda       = 4.0,
    reg_alpha        = 0.5,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# Class labels for display
CLASS_LABELS = {0: "GO-1 High", 1: "GO-2 Medium", 2: "GO-3 Low"}
CLASS_COLORS = {0: PAL["dark_blue"], 1: PAL["orange"], 2: PAL["red"]}

print("Multiclass XGBoost configs:")
print("  XGB_PARAMS_MC          : baseline multiclass (softprob)")
print("  XGB_PARAMS_MC_REG      : anti-overfitting multiclass")
print("  XGB_PARAMS_BINARY_COLLAPSE: GO-3 vs rest (binary, for M7 comparison)")
print("  CLASS_LABELS:", CLASS_LABELS)


## 3. Data Loading — WASI K=3 Clusters

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# Data loading for WASI K=3 clustering
#
# Differences from M7:
#   cluster_path → clusters_GO_WASI_v1.csv  (K=3, labels 1/2/3)
#   GO_i values  → 1 (High), 2 (Medium), 3 (Low)
#   Remapping    → 0/1/2 for XGBoost compatibility
#   No binary filter on GO_i (all three classes kept)
# ─────────────────────────────────────────────────────────────────────────

# ── Paths — adjust to your environment ───────────────────────────────────
DATA_MASTER   = "/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv"
DATA_CLUSTER  = "/Volumes/workspace/default/kmc_data/clusters_GO_WASI_k3.csv"   # WASI K=3 output
MLFLOW_URI    = "mlruns"
EXP_NAME      = "SavingBrains_M8_WASI_K3"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXP_NAME)


def load_cohort_k3(master_path: str, cluster_path: str) -> pd.DataFrame:
    """
    Load master dataset and merge WASI K=3 cluster labels.

    WASI K=3 label mapping
    ----------------------
    CSV value 1 → GO-1 High IQ   → internal class 0
    CSV value 2 → GO-2 Medium IQ → internal class 1
    CSV value 3 → GO-3 Low IQ    → internal class 2

    XGBoost requires class labels starting at 0 for multiclass.
    The original CSV labels (1/2/3) are kept in GO_i_orig for reference.
    GO_i_mc contains the remapped labels (0/1/2).
    GO_i_binary = 1 if GO-3 Low (risk), 0 otherwise (for binary collapse).

    Returns
    -------
    pd.DataFrame  G1 preterm (KMC + TC, preterm==1), all three GO-i classes
    """
    df = pd.read_csv(master_path, low_memory=False)
    cl = pd.read_csv(cluster_path)[["code", "GO_i"]]
    df = df.merge(cl, on="code", how="inner")

    to_n  = lambda s: pd.to_numeric(s, errors="coerce")
    grupo = to_n(df["ubicac"])
    pret  = to_n(df["preterm"])
    mask  = grupo.isin([1, 2]) & (pret == 1) & df["GO_i"].notna()
    df_out = df[mask].copy().reset_index(drop=True)

    # Remap: 1→0, 2→1, 3→2  (XGBoost requires 0-indexed classes)
    df_out["GO_i_orig"]   = df_out["GO_i"].astype(int)
    df_out["GO_i_mc"]     = df_out["GO_i_orig"] - 1
    df_out["GO_i_binary"] = (df_out["GO_i_orig"] == 3).astype(int)

    return df_out


df_g1 = load_cohort_k3(DATA_MASTER, DATA_CLUSTER)
grupo_g1 = pd.to_numeric(df_g1["ubicac"], errors="coerce")

n_total = len(df_g1)
counts  = df_g1["GO_i_orig"].value_counts().sort_index()
print("WASI K=3 cohort loaded:")
print(f"  Total : {n_total}")
for lbl, cls in [(1,"GO-1 High"),(2,"GO-2 Medium"),(3,"GO-3 Low")]:
    n = int(counts.get(lbl, 0))
    print(f"  {cls:<14}: n={n} ({100*n/n_total:.1f}%)")
print()
print("Binary collapse (GO-3 vs rest):")
n_go3  = int((df_g1["GO_i_binary"] == 1).sum())
n_rest = int((df_g1["GO_i_binary"] == 0).sum())
print(f"  GO-3 Low (risk)  : n={n_go3} ({100*n_go3/n_total:.1f}%)")
print(f"  Rest (GO-1+GO-2) : n={n_rest} ({100*n_rest/n_total:.1f}%)")


## 4. Feature Engineering

In [0]:
def fenton_z(value_cm: float, ga_weeks: float,
             M_table: dict, S_table: dict) -> float:
    """
    Fenton 2013 HC z-score using LMS method (L≈1).
    z = (X / M - 1) / S
    Valid for GA 22–41 weeks only.
    """
    if pd.isna(value_cm) or pd.isna(ga_weeks):
        return np.nan
    ga_int = int(round(ga_weeks))
    if ga_int not in M_table:
        return np.nan
    return (value_cm / M_table[ga_int] - 1) / S_table[ga_int]


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive all engineered features and apply special imputations.

    Special imputation rules
    ------------------------
    EX41_durPCconcontroles03 (KMC duration):
        TC group (ubicac==2) → 0.0 by study design.
        TC patients did not receive kangaroo care; their NaN values
        represent structural zeros, not missing data.
        KMC group: raw values retained.

    Fenton HC z-scores
    ------------------
    F_z_hc_birth_fenton   : HC z-score at birth (GA-adjusted, Fenton 2013)
    F_z_hc_40w_fenton     : HC z-score at 40 corrected weeks
    F_catchup_hc_fenton   : z_40w - z_birth  (neonatal brain growth proxy)
                            > 0 = improved position vs GA peers
                            < 0 = lost position vs GA peers (risk)

    Growth velocity (WHO z-scores, corrected age — NIHS reference)
    ---------------------------------------------------------------
    Using velocity (delta z-score) instead of level z-scores avoids
    multicollinearity (r=0.56-0.88 between longitudinal levels).

    F_delta_waz_40w_3m  : Δ weight-for-age z  40w→3m CA
    F_delta_haz_40w_3m  : Δ height-for-age z  40w→3m CA
    F_delta_hcz_40w_3m  : Δ HC-for-age z      40w→3m CA
    F_delta_waz_3m_12m  : Δ weight-for-age z  3m→12m CA  (* p=0.029)
    F_delta_haz_3m_12m  : Δ height-for-age z  3m→12m CA  (* p=0.046)
    F_delta_hcz_3m_12m  : Δ HC-for-age z      3m→12m CA
    """
    df = df.copy()
    to_n = lambda s: pd.to_numeric(s, errors="coerce")
    grupo = to_n(df["ubicac"])

    # ── KMC duration: TC group → 0 by design ─────────────────────────────
    kmc_raw = to_n(df["EX41_durPCconcontroles03"])
    df["EX41_durPCconcontroles03"] = np.where(grupo == 2, 0.0, kmc_raw)

    # ── Fenton HC z-scores ────────────────────────────────────────────────
    ga_birth = to_n(df["BIRTH_ballar5"])
    hc_birth = to_n(df["BIRTH_pc5"]) / 10.0   # mm -> cm
    hc_40w   = to_n(df["EX41_pc8"])            # already cm

    df["F_z_hc_birth_fenton"] = [
        fenton_z(v, e, FENTON_HC_M, FENTON_HC_S)
        for v, e in zip(hc_birth, ga_birth)
    ]
    df["F_z_hc_40w_fenton"] = [
        fenton_z(v, 40, FENTON_HC_M, FENTON_HC_S)
        for v in hc_40w
    ]
    df["F_catchup_hc_fenton"] = (
        df["F_z_hc_40w_fenton"] - df["F_z_hc_birth_fenton"]
    )

    # ── Growth velocity (WHO z-scores with corrected age) ─────────────────
    # Source: NUT12M_wam/ham/pcam at 41w, 3m, 12m CA (NIHS reference)
    wam8  = to_n(df["NUT12M_wam8"])    # WAZ at 41 weeks
    wam9  = to_n(df["NUT12M_wam9"])    # WAZ at 3m CA
    wam12 = to_n(df["NUT12M_wam12"])   # WAZ at 12m CA
    ham8  = to_n(df["NUT12M_ham8"])    # HAZ at 41 weeks
    ham9  = to_n(df["NUT12M_ham9"])    # HAZ at 3m CA
    ham12 = to_n(df["NUT12M_ham12"])   # HAZ at 12m CA
    pcam8 = to_n(df["NUT12M_pcam8"])   # HCZ at 41 weeks
    pcam9 = to_n(df["NUT12M_pcam9"])   # HCZ at 3m CA
    pcam12= to_n(df["NUT12M_pcam12"])  # HCZ at 12m CA

    df["F_delta_waz_40w_3m"]  = wam9  - wam8
    df["F_delta_haz_40w_3m"]  = ham9  - ham8
    df["F_delta_hcz_40w_3m"]  = pcam9 - pcam8
    df["F_delta_waz_3m_12m"]  = wam12 - wam9
    df["F_delta_haz_3m_12m"]  = ham12 - ham9
    df["F_delta_hcz_3m_12m"]  = pcam12 - pcam9

    return df


df_g1 = build_features(df_g1)

# ── Validation ────────────────────────────────────────────────────────────
print("KMC duration imputation check:")
to_n = lambda s: pd.to_numeric(s, errors="coerce")
grupo_g1 = to_n(df_g1["ubicac"])
kmc_col  = to_n(df_g1["EX41_durPCconcontroles03"])
print("  KMC group mean :", round(kmc_col[grupo_g1==1].mean(), 2))
print("  TC  group mean :", round(kmc_col[grupo_g1==2].mean(), 2), "(should be 0.0)")
print("  TC  group NaN  :", int(kmc_col[grupo_g1==2].isna().sum()), "(should be 0)")

print()
print("Fenton features:")
go = df_g1["GO_i"]
for col in ["F_catchup_hc_fenton", "F_z_hc_birth_fenton"]:
    pct = df_g1[col].notna().mean() * 100
    m1  = df_g1.loc[go==1, col].mean()
    m2  = df_g1.loc[go==2, col].mean()
    print(f"  {col:<30} {pct:.0f}% | GO-1={m1:.3f}  GO-2={m2:.3f}")

print()
print("Velocity features (multicollinearity check — expect r < 0.50):")
vel_cols = ["F_delta_waz_40w_3m", "F_delta_waz_3m_12m",
            "F_delta_haz_40w_3m", "F_delta_haz_3m_12m"]
print(df_g1[vel_cols].corr().round(2).to_string())


## 5. Univariate Screening — Kruskal-Wallis + Mann-Whitney

In [0]:
from scipy.stats import kruskal

def univariate_screen_k3(df: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    """
    Univariate screening for three-class target (WASI K=3).

    Tests
    -----
    Kruskal-Wallis H: omnibus test across GO-1 / GO-2 / GO-3.
    Mann-Whitney U: pairwise GO-1 vs GO-3 (most clinically relevant contrast).

    Returns
    -------
    pd.DataFrame with KW p-value + pairwise GO-1 vs GO-3 p-value, FDR-corrected.
    """
    rows = []
    go = df["GO_i_orig"]

    for col in feature_cols:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        pct = s.notna().mean() * 100

        s1 = s[go == 1].dropna()
        s2 = s[go == 2].dropna()
        s3 = s[go == 3].dropna()

        # Omnibus
        try:
            _, p_kw = kruskal(s1, s2, s3)
        except Exception:
            p_kw = np.nan

        # Pairwise GO-1 vs GO-3 (most clinically relevant)
        try:
            _, p_13 = mannwhitneyu(s1, s3, alternative="two-sided")
        except Exception:
            p_13 = np.nan

        rows.append({
            "feature"    : col,
            "pct_data"   : round(pct, 1),
            "mean_go1"   : round(s1.mean(), 3),
            "mean_go2"   : round(s2.mean(), 3),
            "mean_go3"   : round(s3.mean(), 3),
            "p_kruskal"  : round(p_kw, 6) if pd.notna(p_kw) else np.nan,
            "p_go1_go3"  : round(p_13, 6) if pd.notna(p_13) else np.nan,
        })

    df_out = pd.DataFrame(rows)

    # FDR correction on Kruskal-Wallis p-values
    _, qs_kw, _, _ = multipletests(df_out["p_kruskal"].fillna(1), method="fdr_bh")
    df_out["q_kruskal"] = qs_kw.round(4)
    df_out["sig_kw"] = df_out["p_kruskal"].apply(
        lambda p: "***" if p < 0.001 else "**" if p < 0.01 else
                  "*" if p < 0.05 else "†" if p < 0.10 else "ns"
    )
    return df_out.sort_values("p_kruskal").reset_index(drop=True)


all_candidates = list(dict.fromkeys(ALL_FEATURES_P1 + ALL_FEATURES_P2))
df_screen = univariate_screen_k3(df_g1, all_candidates)

print("Univariate screening — Kruskal-Wallis across GO-1 / GO-2 / GO-3")
sep = "-" * 75
print(sep)
print(df_screen[["feature","pct_data","mean_go1","mean_go2","mean_go3",
                  "p_kruskal","sig_kw"]].to_string(index=False))
print(sep)
print("Significant KW p<0.05:", (df_screen["p_kruskal"] < 0.05).sum())


## 6. Core ML Functions — Multiclass + Binary Collapse

In [0]:
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    brier_score_loss, roc_curve, confusion_matrix,
    classification_report, average_precision_score,
)

def prepare_arrays_k3(df: pd.DataFrame, feature_cols: list,
                      mode: str = "multiclass") -> tuple:
    """
    Extract feature matrix X and target y for WASI K=3.

    Parameters
    ----------
    mode : "multiclass"  → y = GO_i_mc  (0/1/2)
           "binary"      → y = GO_i_binary (GO-3=1 vs rest=0)
    """
    to_n = lambda s: pd.to_numeric(s, errors="coerce")
    cols = [c for c in feature_cols if c in df.columns]
    X    = df[cols].apply(to_n).values.astype(float)
    y    = (df["GO_i_mc"].values if mode == "multiclass"
            else df["GO_i_binary"].values)
    return X, y, cols


def cross_validate_mc(
    X: np.ndarray, y: np.ndarray,
    params: dict, mode: str = "multiclass",
    n_folds: int = 5, random_state: int = 42,
) -> dict:
    """
    Stratified K-Fold CV for multiclass or binary-collapse target.

    Metrics (multiclass mode)
    -------------------------
    - macro-AUC (one-vs-rest, requires softprob output)
    - weighted F1
    - per-class precision, recall
    - 3x3 confusion matrix

    Metrics (binary mode)
    ---------------------
    - AUC-ROC, PR-AUC, F1-optimal threshold
    - Sensitivity, Specificity, FN, FP
    """
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import label_binarize

    skf       = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
    n_classes = 3 if mode == "multiclass" else 2
    oof_probs = np.zeros((len(y), n_classes))
    train_aucs = []

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        sc   = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

        clf = XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        p_te = clf.predict_proba(X_te)
        p_tr = clf.predict_proba(X_tr)

        # For binary softprob XGBoost returns shape (n, 2)
        if mode == "multiclass" and p_te.ndim == 1:
            # softmax returns 1-D when n_class param is set
            p_te = p_te.reshape(-1, n_classes)
            p_tr = p_tr.reshape(-1, n_classes)

        oof_probs[te_idx] = p_te

        if mode == "multiclass":
            y_bin = label_binarize(y_tr, classes=[0, 1, 2])
            train_aucs.append(roc_auc_score(y_bin, p_tr, multi_class="ovr",
                                            average="macro"))
        else:
            train_aucs.append(roc_auc_score(y_tr, p_tr[:, 1]))

    # ── Compute metrics ───────────────────────────────────────────────────
    if mode == "multiclass":
        y_pred = np.argmax(oof_probs, axis=1)
        y_bin  = label_binarize(y, classes=[0, 1, 2])
        macro_auc = roc_auc_score(y_bin, oof_probs, multi_class="ovr",
                                  average="macro")
        w_f1  = f1_score(y, y_pred, average="weighted", zero_division=0)
        mac_f1= f1_score(y, y_pred, average="macro",    zero_division=0)
        cm    = confusion_matrix(y, y_pred)
        report= classification_report(y, y_pred,
                    target_names=list(CLASS_LABELS.values()), zero_division=0)
        metrics = {
            "macro_auc"   : float(macro_auc),
            "weighted_f1" : float(w_f1),
            "macro_f1"    : float(mac_f1),
            "auc_train"   : float(np.mean(train_aucs)),
            "gap"         : float(np.mean(train_aucs)) - float(macro_auc),
            "confusion"   : cm,
            "report"      : report,
            "oof_probs"   : oof_probs,
        }
    else:
        # Binary collapse: GO-3 vs rest
        oof_binary = oof_probs[:, 1]
        auc_oof    = roc_auc_score(y, oof_binary)
        pr_auc     = average_precision_score(y, oof_binary)
        thresholds = np.linspace(0.05, 0.95, 181)
        f1s        = [f1_score(y, (oof_binary >= t).astype(int), zero_division=0)
                      for t in thresholds]
        best_thr   = float(thresholds[np.argmax(f1s)])
        pred       = (oof_binary >= best_thr).astype(int)
        cm         = confusion_matrix(y, pred)
        tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
        metrics = {
            "auc_oof"       : float(auc_oof),
            "pr_auc"        : float(pr_auc),
            "auc_train"     : float(np.mean(train_aucs)),
            "gap"           : float(np.mean(train_aucs)) - float(auc_oof),
            "f1"            : float(f1_score(y, pred, zero_division=0)),
            "sensitivity"   : float(recall_score(y, pred, zero_division=0)),
            "specificity"   : float(recall_score(1-y, 1-pred, zero_division=0)),
            "precision_ppv" : float(precision_score(y, pred, zero_division=0)),
            "best_threshold": best_thr,
            "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
            "fn_rate": float(fn/(tp+fn)) if (tp+fn) > 0 else float("nan"),
            "oof_probs": oof_binary,
        }

    return metrics


def fit_final_model_k3(X: np.ndarray, y: np.ndarray, params: dict) -> dict:
    """Fit on full dataset for SHAP and serialisation."""
    imp   = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X)
    sc    = StandardScaler()
    X_sc  = sc.fit_transform(X_imp)
    clf   = XGBClassifier(**params)
    clf.fit(X_sc, y)
    return {"clf": clf, "imputer": imp, "scaler": sc, "X_scaled": X_sc}


print("Functions ready: prepare_arrays_k3, cross_validate_mc, fit_final_model_k3")


## 7. MLflow Experiment Runner — WASI K=3

In [0]:
def run_experiment_k3(
    run_name: str, feature_cols: list, df: pd.DataFrame,
    xgb_params: dict, mode: str = "multiclass",
    tags: dict = None, n_folds: int = 5, random_state: int = 42,
) -> dict:
    """
    MLflow run for WASI K=3 model.

    Parameters
    ----------
    mode : "multiclass"  → GO-1/GO-2/GO-3 simultaneously
           "binary"      → GO-3 vs rest (for M7 comparison)
    """
    X, y, used_cols = prepare_arrays_k3(df, feature_cols, mode=mode)
    if len(used_cols) < len(feature_cols):
        dropped = set(feature_cols) - set(used_cols)
        print("  WARNING — features not found:", dropped)

    with mlflow.start_run(run_name=run_name) as run:
        default_tags = {
            "project"       : "SavingBrains",
            "dataset"       : "KMC-400-20y",
            "model"         : "M8_WASI_K3",
            "clustering"    : "WASI_K3",
            "mode"          : mode,
            "n_classes"     : "3" if mode == "multiclass" else "2 (GO-3 vs rest)",
        }
        if tags:
            default_tags.update(tags)
        mlflow.set_tags(default_tags)

        mlflow.log_params({
            "n_features"  : len(used_cols),
            "n_folds"     : n_folds,
            "mode"        : mode,
            "random_state": random_state,
            **{"xgb_" + k: v for k, v in xgb_params.items()},
        })

        m = cross_validate_mc(X, y, xgb_params, mode, n_folds, random_state)

        # Log mode-specific metrics
        if mode == "multiclass":
            mlflow.log_metrics({
                "macro_auc"   : round(m["macro_auc"], 4),
                "weighted_f1" : round(m["weighted_f1"], 4),
                "macro_f1"    : round(m["macro_f1"], 4),
                "auc_train"   : round(m["auc_train"], 4),
                "gap"         : round(m["gap"], 4),
            })
        else:
            mlflow.log_metrics({
                "auc_oof"       : round(m["auc_oof"], 4),
                "pr_auc"        : round(m["pr_auc"], 4),
                "f1"            : round(m["f1"], 4),
                "sensitivity"   : round(m["sensitivity"], 4),
                "specificity"   : round(m["specificity"], 4),
                "auc_train"     : round(m["auc_train"], 4),
                "gap"           : round(m["gap"], 4),
                "FN"            : m["FN"], "FP": m["FP"],
                "fn_rate"       : round(m["fn_rate"], 4),
            })

        final = fit_final_model_k3(X, y, xgb_params)
        mlflow.xgboost.log_model(final["clf"], artifact_path="model")
        run_id = run.info.run_id

    sep = "-" * 60
    print("Run:", run_name, "| mode=" + mode, "| ID:", run_id[:8] + "...")
    print(sep)
    if mode == "multiclass":
        print("  Macro-AUC  :", round(m["macro_auc"], 4))
        print("  Weighted F1:", round(m["weighted_f1"], 4))
        print("  Gap        :", round(m["gap"], 4))
        print("  Confusion matrix:")
        print(m["confusion"])
    else:
        print("  AUC-OOF    :", round(m["auc_oof"], 4))
        print("  PR-AUC     :", round(m["pr_auc"], 4))
        print("  F1         :", round(m["f1"], 4),
              "  thr=" + str(round(m["best_threshold"], 2)))
        print("  Sensitivity:", round(m["sensitivity"], 4))
        print("  Specificity:", round(m["specificity"], 4))
        cm_str = "TN=" + str(m["TN"]) + " FP=" + str(m["FP"]) + " FN=" + str(m["FN"]) + " TP=" + str(m["TP"])
        print(" ", cm_str)
    print(sep)

    return {
        "run_id"      : run_id, "run_name": run_name,
        "mode"        : mode, "metrics": m,
        "feature_cols": used_cols, "final_model": final,
    }


print("run_experiment_k3() ready.")


## 8. Phase 1 Experiments — Birth → 3 Months CA

| Run | Mode | Feature set | Purpose |
|-----|------|-------------|---------|
| `P1_mc_full` | Multiclass | All Phase 1 | Baseline K=3 |
| `P1_mc_reg` | Multiclass | All Phase 1 | Anti-overfitting |
| `P1_bin_full` | Binary (GO-3 vs rest) | All Phase 1 | M7 comparison |


In [0]:
results_p1 = {}
header = "=" * 60

print(header)
print("PHASE 1 — Birth to 3 months CA  |  WASI K=3")
print(header)

# Multiclass baseline
print()
print("Run 1/3 — P1_mc_full (multiclass, all Phase 1 features)")
results_p1["P1_mc_full"] = run_experiment_k3(
    run_name="P1_mc_full", feature_cols=ALL_FEATURES_P1,
    df=df_g1, xgb_params=XGB_PARAMS_MC, mode="multiclass",
    tags={"phase": "1", "experiment_type": "multiclass_baseline"},
)

# Multiclass regularised
print()
print("Run 2/3 — P1_mc_reg (multiclass, regularised)")
results_p1["P1_mc_reg"] = run_experiment_k3(
    run_name="P1_mc_reg", feature_cols=ALL_FEATURES_P1,
    df=df_g1, xgb_params=XGB_PARAMS_MC_REG, mode="multiclass",
    tags={"phase": "1", "experiment_type": "multiclass_regularised"},
)

# Binary collapse — for M7 comparison
print()
print("Run 3/3 — P1_bin_full (GO-3 vs rest, binary collapse)")
results_p1["P1_bin_full"] = run_experiment_k3(
    run_name="P1_bin_full", feature_cols=ALL_FEATURES_P1,
    df=df_g1, xgb_params=XGB_PARAMS_BINARY_COLLAPSE, mode="binary",
    tags={"phase": "1", "experiment_type": "binary_collapse_go3_vs_rest"},
)

print()
print("Phase 1 complete.")


## 9. Phase 2 Experiments — Birth → 12 Months CA

| Run | Mode | Feature set | Purpose |
|-----|------|-------------|---------|
| `P2_mc_full` | Multiclass | All Phase 2 | Baseline K=3 |
| `P2_mc_reg` | Multiclass | Top-15 SHAP | Anti-overfitting |
| `P2_bin_full` | Binary | All Phase 2 | M7 comparison |
| `P2_bin_reg` | Binary | Top-15 SHAP | Best binary |


In [0]:
results_p2 = {}

print(header)
print("PHASE 2 — Birth to 12 months CA  |  WASI K=3")
print(header)

# Multiclass baseline
print()
print("Run 1/4 — P2_mc_full (multiclass, all Phase 2 features)")
results_p2["P2_mc_full"] = run_experiment_k3(
    run_name="P2_mc_full", feature_cols=ALL_FEATURES_P2,
    df=df_g1, xgb_params=XGB_PARAMS_MC, mode="multiclass",
    tags={"phase": "2", "experiment_type": "multiclass_baseline"},
)

# Top-15 features from M7 SHAP (most stable predictors)
FEATURES_TOP15 = [
    "F_delta_waz_3m_12m", "SCB_nivm1", "F_catchup_hc_fenton",
    "PMD_coaudl6", "PMD_RSM6", "SCB_percap1", "EX41_talla8",
    "NEO_fotote6", "PMD_coloco12", "NEO_totoxidias",
    "F_z_hc_birth_fenton", "F_delta_haz_3m_12m", "SCB_nivp1",
    "NEO_HOSP", "PMD_cogrif6",
]

# Multiclass regularised — top 15
print()
print("Run 2/4 — P2_mc_reg (multiclass, top-15 SHAP + regularised)")
results_p2["P2_mc_reg"] = run_experiment_k3(
    run_name="P2_mc_reg", feature_cols=FEATURES_TOP15,
    df=df_g1, xgb_params=XGB_PARAMS_MC_REG, mode="multiclass",
    tags={"phase": "2", "experiment_type": "multiclass_regularised_top15"},
)

# Binary collapse — full Phase 2
print()
print("Run 3/4 — P2_bin_full (GO-3 vs rest, all Phase 2 features)")
results_p2["P2_bin_full"] = run_experiment_k3(
    run_name="P2_bin_full", feature_cols=ALL_FEATURES_P2,
    df=df_g1, xgb_params=XGB_PARAMS_BINARY_COLLAPSE, mode="binary",
    tags={"phase": "2", "experiment_type": "binary_collapse_baseline"},
)

# Binary collapse regularised — top 15 (for M7 comparison)
print()
print("Run 4/4 — P2_bin_reg (GO-3 vs rest, top-15 regularised)")
results_p2["P2_bin_reg"] = run_experiment_k3(
    run_name="P2_bin_reg", feature_cols=FEATURES_TOP15,
    df=df_g1, xgb_params=XGB_PARAMS_BINARY_COLLAPSE, mode="binary",
    tags={"phase": "2", "experiment_type": "binary_collapse_regularised"},
)

print()
print("Phase 2 complete.")


## 10. Cross-Phase Comparison — Multiclass vs Binary vs M7

In [0]:
all_results = {**results_p1, **results_p2}

rows_mc  = []   # multiclass results
rows_bin = []   # binary collapse results

for name, res in all_results.items():
    m = res["metrics"]
    phase = "1" if name.startswith("P1") else "2"

    if res["mode"] == "multiclass":
        rows_mc.append({
            "Run"         : name,
            "Phase"       : phase,
            "N_features"  : len(res["feature_cols"]),
            "Macro_AUC"   : round(m["macro_auc"], 4),
            "Weighted_F1" : round(m["weighted_f1"], 4),
            "Macro_F1"    : round(m["macro_f1"], 4),
            "Gap"         : round(m["gap"], 4),
        })
    else:
        rows_bin.append({
            "Run"         : name,
            "Phase"       : phase,
            "N_features"  : len(res["feature_cols"]),
            "AUC_OOF"     : round(m["auc_oof"], 4),
            "PR_AUC"      : round(m["pr_auc"], 4),
            "F1"          : round(m["f1"], 4),
            "Sensitivity" : round(m["sensitivity"], 4),
            "Specificity" : round(m["specificity"], 4),
            "FN"          : m["FN"], "FP": m["FP"],
            "Gap"         : round(m["gap"], 4),
        })

df_mc  = pd.DataFrame(rows_mc).sort_values("Macro_AUC",  ascending=False)
df_bin = pd.DataFrame(rows_bin).sort_values("AUC_OOF",   ascending=False)

print("── MULTICLASS (GO-1 / GO-2 / GO-3) ──")
print(df_mc.to_string(index=False))
print()
print("── BINARY COLLAPSE (GO-3 vs rest) ──")
print(df_bin.to_string(index=False))

# Select best run per mode
best_mc_name  = df_mc.iloc[0]["Run"]  if len(df_mc)  > 0 else None
best_bin_name = df_bin.iloc[0]["Run"] if len(df_bin) > 0 else None
best_mc  = all_results[best_mc_name]  if best_mc_name  else None
best_bin = all_results[best_bin_name] if best_bin_name else None

print()
print("Best multiclass run  :", best_mc_name)
print("Best binary run      :", best_bin_name)
print()

# ── Comparison chart (inline) ─────────────────────────────────────────────
fig, (ax_mc, ax_bin) = plt.subplots(1, 2, figsize=(13, 4))

if len(df_mc) > 0:
    ax_mc.barh(df_mc["Run"], df_mc["Macro_AUC"],
               color=PAL["dark_blue"], edgecolor="white", alpha=0.85)
    ax_mc.axvline(1/3, color="red", ls="--", lw=1, alpha=0.6,
                  label="Chance (1/3)")
    ax_mc.set_xlim(0.25, 0.80)
    ax_mc.set_title("Multiclass — Macro-AUC\nWASI K=3 (GO-1/GO-2/GO-3)",
                    fontweight="bold", fontsize=11)
    ax_mc.legend(fontsize=8)
    for bar, val in zip(ax_mc.patches, df_mc["Macro_AUC"]):
        ax_mc.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                   f"{val:.3f}", va="center", fontsize=9, fontweight="bold")

if len(df_bin) > 0:
    ax_bin.barh(df_bin["Run"], df_bin["AUC_OOF"],
                color=PAL["orange"], edgecolor="white", alpha=0.85)
    ax_bin.axvline(0.5, color="red", ls="--", lw=1, alpha=0.6,
                   label="Chance (0.5)")
    # M7 reference line
    ax_bin.axvline(0.678, color=PAL["mid_blue"], ls=":", lw=1.5,
                   alpha=0.8, label="M7 best (0.678)")
    ax_bin.set_xlim(0.40, 0.80)
    ax_bin.set_title("Binary collapse — AUC-OOF\nGO-3 Low vs (GO-1+GO-2)",
                     fontweight="bold", fontsize=11)
    ax_bin.legend(fontsize=8)
    for bar, val in zip(ax_bin.patches, df_bin["AUC_OOF"]):
        ax_bin.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                    f"{val:.3f}", va="center", fontsize=9, fontweight="bold")

fig.suptitle("M8 WASI K=3 — Experiment comparison\n"
             "Saving Brains · G1 preterm n=386",
             fontweight="bold", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## 11. Best Model Analysis

In [0]:
# ── 11a. Multiclass confusion matrix + per-class metrics ──────────────────
if best_mc is not None:
    m_mc = best_mc["metrics"]
    print("Best multiclass model:", best_mc_name)
    print()
    print("Classification report:")
    print(m_mc["report"])
    print()

    cm_mc = m_mc["confusion"]
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_mc, cmap="Blues")
    for i in range(3):
        for j in range(3):
            v = cm_mc[i, j]
            color = "white" if v > cm_mc.max() * 0.55 else "black"
            ax.text(j, i, str(v), ha="center", va="center",
                    fontsize=13, fontweight="bold", color=color)
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(["Pred GO-1\nHigh", "Pred GO-2\nMedium",
                         "Pred GO-3\nLow"], fontsize=10)
    ax.set_yticklabels(["True GO-1\nHigh", "True GO-2\nMedium",
                         "True GO-3\nLow"], fontsize=10)
    ax.set_title("Confusion Matrix 3×3\n" + best_mc_name,
                 fontweight="bold", fontsize=11)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


In [0]:
# ── 11b. Binary collapse — metrics panel + ROC (M7 style) ────────────────
if best_bin is not None:
    m_bin = best_bin["metrics"]
    oof_b = m_bin["oof_probs"]
    thr_str = str(round(m_bin["best_threshold"], 2))
    auc_str = str(round(m_bin["auc_oof"], 4))
    tn, fp, fn, tp = m_bin["TN"], m_bin["FP"], m_bin["FN"], m_bin["TP"]
    _, y_bin_arr, _ = prepare_arrays_k3(df_g1, best_bin["feature_cols"],
                                         mode="binary")

    print("Best binary model:", best_bin_name)
    print("AUC-OOF:", auc_str,
          "  F1:", round(m_bin["f1"], 4),
          "  thr:", thr_str)

    fig = plt.figure(figsize=(13, 5))
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

    # Metrics bar
    ax1 = fig.add_subplot(gs[0])
    m_names = ["AUC-ROC", "F1-Score", "Sensitivity",
               "Specificity", "Precision"]
    m_vals  = [m_bin["auc_oof"], m_bin["f1"], m_bin["sensitivity"],
               m_bin["specificity"], m_bin["precision_ppv"]]
    m_cols  = [PAL["dark_blue"], PAL["mid_blue"], PAL["green"],
               PAL["red"], PAL["light_blue"]]
    bars = ax1.barh(m_names, m_vals, color=m_cols, edgecolor="white", alpha=0.88)
    ax1.set_xlim(0, 1.14)
    ax1.axvline(0.5, color="grey", ls="--", lw=1, alpha=0.5)
    for bar, val in zip(bars, m_vals):
        ax1.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                 f"{val:.3f}", va="center", fontsize=9.5, fontweight="bold")
    ax1.set_title("Metrics — " + best_bin_name + "\n(GO-3 vs rest, thr=" + thr_str + ")",
                  fontweight="bold", fontsize=11)

    # Confusion matrix
    ax2 = fig.add_subplot(gs[1])
    cm2 = np.array([[tn, fp], [fn, tp]])
    ax2.imshow(cm2, cmap="Blues", vmin=0, vmax=cm2.max() * 1.3)
    for i, row in enumerate([["TN", "FP"], ["FN", "TP"]]):
        for j, lbl in enumerate(row):
            v = cm2[i, j]
            color = "white" if v > cm2.max() * 0.55 else "black"
            ax2.text(j, i, lbl + "\n" + str(v),
                     ha="center", va="center", fontsize=13,
                     fontweight="bold", color=color)
    ax2.set_xticks([0, 1])
    ax2.set_yticks([0, 1])
    ax2.set_xticklabels(["Pred rest\n(GO-1+GO-2)", "Pred GO-3\nLow"], fontsize=9)
    ax2.set_yticklabels(["True rest", "True GO-3\nLow"], fontsize=9)
    ax2.set_title("Confusion Matrix\nFN=" + str(fn) + "  FP=" + str(fp),
                  fontweight="bold", fontsize=11)

    # ROC curve
    ax3 = fig.add_subplot(gs[2])
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_bin_arr, oof_b)
    ax3.plot(fpr, tpr, lw=2.5, color=PAL["orange"],
             label="M8 WASI K=3 (AUC=" + auc_str + ")")
    ax3.axvline(0, ymin=0, ymax=1)
    ax3.plot([0,1],[0,1],"k--", lw=1, alpha=0.4, label="Chance")
    ax3.axhline(0.678, color=PAL["dark_blue"], ls=":", lw=1.5,
                label="M7 best AUC=0.678", alpha=0.7)
    ax3.set_xlabel("1 - Specificity (FPR)", fontsize=10)
    ax3.set_ylabel("Sensitivity (TPR)", fontsize=10)
    ax3.set_title("ROC — GO-3 Low prediction\nvs M7 reference",
                  fontweight="bold", fontsize=11)
    ax3.legend(fontsize=8, loc="lower right")

    fig.suptitle("M8 WASI K=3 — Binary collapse (GO-3 vs rest)",
                 fontweight="bold", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


## 12. SHAP — Feature Importance (Best Binary Model)

In [0]:
# SHAP analysis on best binary collapse model (most interpretable for clinicians)
import shap
import math

if best_bin is not None:
    final_md = best_bin["final_model"]
    clf      = final_md["clf"]
    imp_f    = final_md["imputer"]
    sc_f     = final_md["scaler"]
    feat_cls = best_bin["feature_cols"]

    to_n    = lambda s: pd.to_numeric(s, errors="coerce")
    cols_ok = [c for c in feat_cls if c in df_g1.columns]
    X_raw   = df_g1[cols_ok].apply(to_n).values.astype(float)
    X_imp   = imp_f.transform(X_raw)
    X_sc    = sc_f.transform(X_imp)
    probs   = clf.predict_proba(X_sc)[:, 1]
    thr     = best_bin["metrics"]["best_threshold"]
    preds   = (probs >= thr).astype(int)
    _, y_b, _ = prepare_arrays_k3(df_g1, feat_cls, mode="binary")

    N_SHAP   = min(180, X_sc.shape[0])
    shap_idx = np.random.RandomState(RANDOM_STATE).choice(X_sc.shape[0],
               N_SHAP, replace=False)
    explainer   = shap.TreeExplainer(clf)
    shap_values = explainer(X_sc[shap_idx], check_additivity=False)
    shap_values.feature_names = cols_ok

    # Select representative patients
    idx_tp = np.where((y_b == 1) & (preds == 1))[0]
    idx_fn = np.where((y_b == 1) & (preds == 0))[0]

    fig, (ax_bee, ax_bar) = plt.subplots(1, 2, figsize=(14, 6))

    plt.sca(ax_bee)
    shap.plots.beeswarm(shap_values, max_display=12, show=False)
    ax_bee.set_title("SHAP Beeswarm — " + best_bin_name + "\n"
                     "GO-3 Low risk prediction",
                     fontweight="bold", fontsize=11)

    mean_shap = np.abs(shap_values.values).mean(axis=0)
    df_shap   = (pd.DataFrame({"feature": cols_ok, "shap_mean": mean_shap})
                 .sort_values("shap_mean", ascending=False)
                 .reset_index(drop=True))
    top15 = df_shap.head(15).sort_values("shap_mean")
    ax_bar.barh(top15["feature"], top15["shap_mean"],
                color=PAL["orange"], edgecolor="white", alpha=0.88)
    ax_bar.set_xlabel("Mean |SHAP| value", fontsize=11)
    ax_bar.set_title("Top-15 feature importance\nGO-3 Low IQ prediction",
                     fontweight="bold", fontsize=11)

    plt.tight_layout()
    plt.show()

    print("Top 10 features by mean |SHAP|:")
    print(df_shap.head(10).to_string(index=False))

    # Waterfall — most confident GO-3 detection
    if len(idx_tp) > 0:
        pt = int(idx_tp[np.argmax(probs[idx_tp])])
        prob_pt = probs[pt]
        title = ("True Positive GO-3 Low  |  p(GO-3)=" +
                 str(round(prob_pt, 3)) +
                 "  [thr=" + str(round(thr, 2)) + "]")
        plt.figure(figsize=(10, 5))
        shap.plots.waterfall(shap_values[pt], max_display=10, show=False)
        plt.title(title, fontsize=10, fontweight="bold", pad=12)
        plt.tight_layout()
        plt.show()


## 13. M7 vs M8 — Direct Comparison

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# Compare M7 (binary GO-i K=2) vs M8 (WASI K=3 binary collapse GO-3 vs rest)
#
# Key question: Does using WASI-only K=3 (High/Medium/Low) as the target
# produce a better or worse predictive model than the full composite GO-i K=2?
#
# Interpretation:
#   M8 AUC > M7 AUC → WASI-only phenotype is more predictable from neonatal vars
#   M8 AUC < M7 AUC → Full composite GO-i captures more signal (multi-domain better)
#   M8 AUC ≈ M7 AUC → Both phenotypes are equally hard to predict
# ─────────────────────────────────────────────────────────────────────────

m7_ref = {
    "Model"       : "M7 — Binary GO-i K=2 (reference)",
    "Target"      : "GO-2 Low (composite WASI+TAP+CVLT+VMI)",
    "AUC_OOF"     : 0.678,
    "PR_AUC"      : 0.468,
    "F1"          : 0.497,
    "Sensitivity" : 0.779,
    "Specificity" : 0.524,
    "FN"          : 25,
    "FP"          : 127,
    "Gap"         : 0.157,
}

print("=" * 70)
print("DIRECT COMPARISON: M7 (GO-i K=2) vs M8 (WASI K=3 binary collapse)")
print("=" * 70)
print()

ref_hdr = f"  {'Metric':<20} {'M7 ref':>10}  {'M8 best':>10}  {'Delta':>10}"
print(ref_hdr)
print("  " + "─" * 58)

if best_bin is not None:
    m8 = best_bin["metrics"]
    pairs = [
        ("AUC-OOF",     m7_ref["AUC_OOF"],     round(m8["auc_oof"], 4)),
        ("PR-AUC",      m7_ref["PR_AUC"],       round(m8["pr_auc"], 4)),
        ("F1-Score",    m7_ref["F1"],            round(m8["f1"], 4)),
        ("Sensitivity", m7_ref["Sensitivity"],   round(m8["sensitivity"], 4)),
        ("Specificity", m7_ref["Specificity"],   round(m8["specificity"], 4)),
        ("FN (missed)", m7_ref["FN"],            m8["FN"]),
        ("FP (alarms)", m7_ref["FP"],            m8["FP"]),
        ("Gap",         m7_ref["Gap"],           round(m8["gap"], 4)),
    ]
    for name, v7, v8 in pairs:
        delta = round(v8 - v7, 4) if isinstance(v8, float) else v8 - v7
        arrow = "▲" if delta > 0 else "▼" if delta < 0 else "─"
        print(f"  {name:<20} {str(v7):>10}  {str(v8):>10}  {arrow} {delta}")

print()
print("Interpretation:")
print("  If M8 AUC > M7: WASI-only phenotype (High/Med/Low) is more predictable")
print("  If M8 AUC < M7: Full composite GO-i (WASI+TAP+CVLT+VMI) adds signal")
print("  Clinically, both targets have value — K=3 adds the 'medium risk' group")
print("  which may benefit from different interventions than Low IQ (GO-3)")


## 14. Save Artefacts & Final Summary

In [0]:
# Save comparison tables
if len(df_mc) > 0:
    df_mc.to_csv("m8_resultados_multiclass.csv", index=False, encoding="utf-8-sig")
if len(df_bin) > 0:
    df_bin.to_csv("m8_resultados_binary.csv", index=False, encoding="utf-8-sig")
df_screen.to_csv("m8_screening_kruskal.csv", index=False, encoding="utf-8-sig")

# Save best binary model (for deployment comparison with M7)
if best_bin is not None:
    bundle = {
        "clf"          : best_bin["final_model"]["clf"],
        "imputer"      : best_bin["final_model"]["imputer"],
        "scaler"       : best_bin["final_model"]["scaler"],
        "feature_cols" : best_bin["feature_cols"],
        "threshold"    : best_bin["metrics"]["best_threshold"],
        "metrics"      : best_bin["metrics"],
        "run_id"       : best_bin["run_id"],
        "target"       : "GO3_Low_WASI_K3",
        "model"        : "M8_WASI_K3_binary_collapse",
    }
    joblib.dump(bundle, "m8_modelo_binary_best.joblib")
    print("Saved: m8_modelo_binary_best.joblib")

print("Saved:")
print("  m8_resultados_multiclass.csv")
print("  m8_resultados_binary.csv")
print("  m8_screening_kruskal.csv")
print()

sep = "=" * 65
print(sep)
print("  FINAL SUMMARY — M8 WASI K=3")
print(sep)
if best_mc is not None:
    print("  Best multiclass  :", best_mc_name)
    print("  Macro-AUC        :", round(best_mc["metrics"]["macro_auc"], 4))
    print("  Weighted F1      :", round(best_mc["metrics"]["weighted_f1"], 4))
if best_bin is not None:
    print()
    print("  Best binary (GO-3 vs rest):", best_bin_name)
    print("  AUC-OOF          :", round(best_bin["metrics"]["auc_oof"], 4))
    print("  F1               :", round(best_bin["metrics"]["f1"], 4))
    print("  FN               :", best_bin["metrics"]["FN"], "(GO-3 Low missed)")
print()
print("  M7 reference AUC : 0.678 (binary GO-i K=2)")
print(sep)
print()
uri_str = mlflow.get_tracking_uri()
print("MLflow URI:", uri_str)
print("Launch UI : mlflow ui --backend-store-uri " + uri_str)
